In [1]:
# the codes in this cell are not important

from multiprocessing import context
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

# calculate the attention scores for the second input

query = inputs[1]  # 2nd input token is the query

attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query) # dot product (transpose not necessary here since they are 1-dim vectors)

print(attn_scores_2)


# calculate the attention weights for the second input
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)

print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())


# calculate the context vector for the second input

query = inputs[1] # 2nd input token is the query

context_vec_2 = torch.zeros(query.shape) 
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i

print(context_vec_2)


## explain the following lines of code

"""

1st line: attn_scores_2 = torch.empty(inputs.shape[0])

This line creates an empty tensor of shape (inputs.shape[0],) to store the attention scores for each input token.

2nd line: context_vec_2 = torch.zeros(query.shape) 

This line creates a tensor of shape (query.shape) to store the context vector for the second input.


"""

## for ALL INPUTS

attn_scores = inputs @ inputs.T

attn_weights = torch.softmax(attn_scores, dim=-1) # what is the last dimension? the last dimension is the number of input tokens.

context_vec = attn_weights @ inputs





/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])
Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)
tensor([0.4419, 0.6515, 0.5683])


In [ ]:
print(attn_weights.shape)
#

torch.Size([6, 6])


In [3]:
## self-attention with trainable weights
import torch.nn as nn

class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

NameError: name 'd_in' is not defined

## first method to apply casual attention (Not recommended)
to hide future words

In [ ]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs) 
attn_scores = queries @ keys.T

attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)



In [ ]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

masked_simple = attn_weights * mask_simple
print(masked_simple)

"""
However, if the mask were applied after softmax, like above, it would disrupt the probability distribution created by softmax
Softmax ensures that all output values sum to 1
Masking after softmax would require re-normalizing the outputs to sum to 1 again, which complicates the process and might lead to unintended effects
"""

row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)


## Second method to apply casual attention (recommended)
to hide future words



In [ ]:
"""
Instead of masking the attention scores after softmax, we can mask the attention scores before softmax
"""

mask = torch.triu(torch.ones(context_length,context_length), diagonal=1)
msked_attn_scores = attn_scores.masked_fill(mask.bool(), -torch.inf)

attn_weights = torch.softmax(msked_attn_scores / keys.shape[-1]**0.5, dim=-1)

print(attn_weights)
